In [44]:
import sys
print(sys.executable)

c:\Users\ASUS\anaconda3\python.exe


In [45]:
!pip install fastapi "uvicorn[standard]" python-multipart jinja2 python-dotenv pydantic langchain langchain-core langchain-community langchain-text-splitters langgraph langgraph-checkpoint-sqlite tavily-python pypdf python-docx langchain-groq langchain-huggingface sentence-transformers langchain-qdrant qdrant-client

Defaulting to user installation because normal site-packages is not writeable


In [46]:
import sys
!{sys.executable} -m pip install fastapi "uvicorn[standard]" python-multipart jinja2 python-dotenv pydantic

In [47]:
import sys
!{sys.executable} -m pip install langchain langchain-core langchain-community langchain-text-splitters langgraph langchain-groq tavily-python pypdf python-docx

In [48]:
import sys
!{sys.executable} -m pip install langchain-huggingface sentence-transformers langchain-qdrant qdrant-client

In [49]:
%pip install --upgrade typing-extensions

Note: you may need to restart the kernel to use updated packages.


In [50]:
packages_to_test = [
    ("langchain_community", "langchain-community"),
    ("langchain_text_splitters", "langchain-text-splitters"),
    ("langchain_core", "langchain-core"),
    ("langchain_groq", "langchain-groq"),
    ("langchain_huggingface", "langchain-huggingface"),
    ("langchain_qdrant", "langchain-qdrant"),
    ("qdrant_client", "qdrant-client"),
    ("langgraph", "langgraph")
]

for module_name, pip_name in packages_to_test:
    try:
        __import__(module_name)
        print(f"✅ {module_name} is installed.")
    except ImportError:
        print(f"❌ MISSING: {module_name} (Run: pip install {pip_name})")

✅ langchain_community is installed.
✅ langchain_text_splitters is installed.
✅ langchain_core is installed.
✅ langchain_groq is installed.
✅ langchain_huggingface is installed.
✅ langchain_qdrant is installed.
✅ qdrant_client is installed.
✅ langgraph is installed.


In [51]:
import sys
import typing_extensions

print("Python:", sys.executable)
print("typing_extensions:", typing_extensions.__file__)
print("TypeForm available:", hasattr(typing_extensions, "TypeForm"))

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from langgraph.graph import StateGraph, START, END

print("All imports succeeded")

Python: c:\Users\ASUS\anaconda3\python.exe
typing_extensions: c:\Users\ASUS\anaconda3\Lib\site-packages\typing_extensions.py
TypeForm available: True
All imports succeeded


In [52]:
from typing import List, TypedDict, Literal
from pydantic import BaseModel, Field

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv

load_dotenv()

True

In [53]:
docs = (
    PyPDFLoader("./documents/RunbookPilot_Company_Profile.pdf").load()
    + PyPDFLoader("./documents/RunbookPilot_Comprehensive_Employee_Handbook.pdf").load()
    + PyPDFLoader("./documents/RunbookPilot_Products_Pricing_Catalog.pdf").load()
)

In [54]:
chunks = RecursiveCharacterTextSplitter(
    chunk_size=600, chunk_overlap=150
).split_documents(docs)

In [55]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_store = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    location=":memory:",
    collection_name="ashish_ai_collection",
)
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [57]:
from langchain_groq import ChatGroq

# Use an active, supported model ID on Groq
llm = ChatGroq(
    model="openai/gpt-oss-20b",  # or "openai/gpt-oss-120b"
    temperature=0
)

In [58]:
# --------------------------------------------------
# Graph State
# --------------------------------------------------
class State(TypedDict):
    question: str
    need_retrieval: bool

    docs: List[Document]

    answer: str

In [59]:
class RetrieveDecision(BaseModel):
    should_retrieve: bool = Field(
        ...,
        description="True if external documents are needed to answer reliably, else False."
    )

decide_retrieval_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You decide whether retrieval is needed.\n"
            "Return JSON that matches this schema:\n"
            "{{'should_retrieve': boolean}}\n\n"
            "Guidelines:\n"
            "- should_retrieve=True if answering requires specific facts, citations, or info likely not in the model.\n"
            "- should_retrieve=False for general explanations, definitions, or reasoning that doesn't need sources.\n"
            "- If unsure, choose True."
        ),
        ("human", "Question: {question}"),
    ]
)


# IMPORTANT: no `.content` for structured output
should_retrieve_llm = llm.with_structured_output(RetrieveDecision)

def decide_retrieval(state: "State"):
    decision: RetrieveDecision = should_retrieve_llm.invoke(
        decide_retrieval_prompt.format_messages(question=state["question"])
    )
    return {"need_retrieval": decision.should_retrieve}

In [60]:
direct_generation_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Answer the question using only your general knowledge.\n"
            "Do NOT assume access to external documents.\n"
            "If you are unsure or the answer requires specific sources, say:\n"
            "'I don't know based on my general knowledge.'"
        ),
        ("human", "{question}"),
    ]
)


def generate_direct(state: State):
    out = llm.invoke(
        direct_generation_prompt.format_messages(
            question=state["question"]
        )
    )
    return {
        "answer": out.content
    }


In [61]:
def retrieve(state: State):
    return {"docs": retriever.invoke(state["question"])}

In [62]:
def route_after_decide(state: State) -> Literal["generate_direct", "retrieve"]:
    if state["need_retrieval"]:
        return "retrieve"
    return "generate_direct"

In [63]:
from langgraph.graph import StateGraph, START, END

# 1. Define the retrieval node function
def retrieve_node(state: State):
    retrieved_docs = retriever.invoke(state["question"])
    return {"docs": retrieved_docs}

# 2. Define the RAG generation node function
def generate_rag(state: State):
    context = "\n\n".join([doc.page_content for doc in state["docs"]])
    rag_prompt = f"Answer the question based only on this context:\n\n{context}\n\nQuestion: {state['question']}"
    out = llm.invoke(rag_prompt)
    return {"answer": out.content}

# 3. Initialize and build the StateGraph
g = StateGraph(State)

# Add all nodes
g.add_node("decide_retrieval", decide_retrieval)
g.add_node("generate_direct", generate_direct)
g.add_node("retrieve", retrieve_node)
g.add_node("generate_rag", generate_rag)

# Add entrypoint and edges
g.add_edge(START, "decide_retrieval")

g.add_conditional_edges(
    "decide_retrieval",
    route_after_decide,
    {
        "generate_direct": "generate_direct",
        "retrieve": "retrieve",
    },
)

g.add_edge("generate_direct", END)
g.add_edge("retrieve", "generate_rag")
g.add_edge("generate_rag", END)

# Compile the final application
app = g.compile()
print("Graph compiled successfully with all nodes defined!")

Graph compiled successfully with all nodes defined!


In [64]:
from langchain_groq import ChatGroq

# 1. Initialize with an active, standard Groq production model
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

# 2. Re-bind the structured output schema for the router
should_retrieve_llm = llm.with_structured_output(RetrieveDecision)

# 3. Re-define the router node using the updated model
def decide_retrieval(state: State):
    decision: RetrieveDecision = should_retrieve_llm.invoke(
        decide_retrieval_prompt.format_messages(question=state["question"])
    )
    return {"need_retrieval": decision.should_retrieve}

# 4. Re-compile your state graph application
app = g.compile()
print("Graph successfully updated and re-compiled with openai/gpt-oss-20b!")

Graph successfully updated and re-compiled with openai/gpt-oss-20b!


In [65]:
result = app.invoke({
    "question": "What is runbookpilot?",
    "need_retrieval": False,
    "docs": [],
    "answer": "",
})

In [66]:

print("Need Retrieval:", result.get('need_retrieval'))
print("Docs:", result.get('docs'))
print("Answer:", result.get('answer'))

Need Retrieval: False
Docs: []
Answer: RunbookPilot is a cloud‑based run‑book management platform designed for IT, DevOps, and SRE teams. It lets teams create, version‑control, and share run‑books (step‑by‑step procedures for troubleshooting, incident response, and routine operations) in a single, searchable knowledge base. Key features typically include:

- **Template library** for common tasks (e.g., reboot a server, roll back a deployment, clear a cache).  
- **Collaboration tools** such as comments, tagging, and real‑time editing.  
- **Version control** so changes are tracked and previous iterations can be restored.  
- **Integration** with incident‑management and chat tools (PagerDuty, Opsgenie, Slack, Teams, etc.) so run‑books can be triggered or referenced during an incident.  
- **Automation hooks** that allow run‑books to be executed automatically via scripts or APIs.  
- **Audit and compliance** reporting to show who edited what and when.

In short, RunbookPilot is meant to 

In [67]:
result['docs']

[]

In [68]:
result['need_retrieval']

False